# FL Project: Optimization Algorithm Comparison
## 1. Data Loading

In [7]:
import sys
sys.path.append("..")  # so Python can find the src/ folder

from src.get_data import fetch_data

df = fetch_data(cache_path="../data/daily_records.csv")
df.head()

Loading cached data from '../data/daily_records.csv'.
  7872 rows, 192 stations, 41 days.


,station,lat,lon,day,tmin,tmax
0,Alajärvi Möksy,63.08898,24.26084,2026-02-20,-25.2,-9.0
1,Alajärvi Möksy,63.08898,24.26084,2026-02-21,-12.5,-2.2
2,Alajärvi Möksy,63.08898,24.26084,2026-02-22,-12.2,-2.0
3,Alajärvi Möksy,63.08898,24.26084,2026-02-23,-20.1,-3.8
4,Alajärvi Möksy,63.08898,24.26084,2026-02-24,-15.7,-4.2


In [8]:
from src.graph import get_station_index, build_knn_graph, graph_summary

stations = get_station_index(df)
A, edge_index = build_knn_graph(stations, k=3, weight="distance")
graph_summary(stations, A, edge_index)

Nodes : 192
Edges : 361
Degree — min: 3  mean: 3.8  max: 7
Weight — min: 0.0029  mean: 0.3960  max: 0.9890


In [9]:
from src.graph import get_station_index
from src.models import build_local_datasets, init_weights, mse, local_gradient

stations = get_station_index(df)
data = build_local_datasets(df, stations)
weights = init_weights(data)

In [10]:
from src.graph import get_station_index, build_knn_graph
from src.models import build_local_datasets, init_weights, mse
from src.fl_algorithm import run_a2, run_a4

stations = get_station_index(df)
A, edge_index = build_knn_graph(stations, k=3, weight="distance")
data = build_local_datasets(df, stations)
w0 = init_weights(data, seed=0)

# A2
h_a2 = run_a2(data, w0, A, alpha=1.0, n_iterations=100)

# A4 with K=1, 2, 3
h_a4_k1 = run_a4(data, w0, A, stations, alpha=1.0, K=1, n_iterations=500)
h_a4_k2 = run_a4(data, w0, A, stations, alpha=1.0, K=2, n_iterations=500)
h_a4_k3 = run_a4(data, w0, A, stations, alpha=1.0, K=3, n_iterations=500)

In [13]:
import numpy as np

a2   = np.load("../results/a2.npz")
a4k1 = np.load("../results/a4_k1.npz")
a4k2 = np.load("../results/a4_k2.npz")
a4k3 = np.load("../results/a4_k3.npz")

# Example: plot val loss vs cumulative messages for all runs
a2["val_loss"]       # array of length 100
a2["cum_messages"]   # array of length 100
a4k1["val_loss"]     # array of length 1000

array([48.42345686, 48.32908449, 48.23088814, 48.12999742, 48.0286352 ,
       47.92813311, 47.82907013, 47.73159603, 47.63565381, 47.54110419,
       47.44778831, 47.35555625, 47.2642779 , 47.17384517, 47.08417042,
       46.99518347, 46.90682838, 46.81906051, 46.73184398, 46.64514968,
       46.55895363, 46.47323576, 46.38797897, 46.30316839, 46.21879086,
       46.13483453, 46.05128859, 45.96814301, 45.88538842, 45.80301599,
       45.72101731, 45.63938437, 45.55810952, 45.47718539, 45.39660491,
       45.31636126, 45.23644789, 45.15685847, 45.07758691, 44.99862734,
       44.91997411, 44.84162175, 44.76356502, 44.68579886, 44.6083184 ,
       44.53111894, 44.45419598, 44.37754516, 44.30116231, 44.22504339,
       44.14918453, 44.07358201, 43.99823222, 43.92313172, 43.84827718,
       43.7736654 , 43.69929329, 43.62515789, 43.55125632, 43.47758584,
       43.40414379, 43.3309276 , 43.2579348 , 43.18516302, 43.11260994,
       43.04027335, 42.96815111, 42.89624115, 42.82454146, 42.75